## mixture laplace t and mad test

In [2]:
function run_ttest_summary(datasets)

    summary = Dict(
        "n" => Int[],
        "TP" => Float64[],
        "FP" => Float64[],
        "Power" => Float64[],
        "FDR" => Float64[]
    )

    # NEW: raw p-values for ranking metrics
    pvals_store =
        Dict{Tuple{Int,Int}, Vector{Float64}}()

    for n in n_values

        println("\nRunning t-test: n = $n")

        power_vals = Float64[]
        fdr_vals   = Float64[]
        tp_vals    = Float64[]
        fp_vals    = Float64[]

        for r in 1:R

            Y =
                datasets[(n,r)].Y

            is_DE =
                datasets[(n,r)].is_DE

            β_hat = [
                mean(Y[g])
                for g in 1:G
            ]

            s_hat = [
                std(Y[g], corrected = true)
                for g in 1:G
            ]

            t_stats =
                sqrt(n) .* β_hat ./ s_hat

            tdist =
                TDist(n - 1)

            # raw p-values
            pvals =
                2 .* ccdf.(
                    Ref(tdist),
                    abs.(t_stats)
                )

            # -------------------------------------------------
            # NEW: save raw p-values for this replicate
            # -------------------------------------------------
            pvals_store[(n,r)] =
                copy(pvals)

            adj_p =
                adjust(
                    pvals,
                    BenjaminiHochberg()
                )

            reject =
                adj_p .<= 0.05

            TP =
                sum(reject .& is_DE)

            FP =
                sum(reject .& .!is_DE)

            power =
                TP / max(sum(is_DE), 1)

            fdp =
                FP / max(sum(reject), 1)

            push!(tp_vals, TP)
            push!(fp_vals, FP)
            push!(power_vals, power)
            push!(fdr_vals, fdp)
        end

        push!(summary["n"], n)
        push!(summary["TP"], mean(tp_vals))
        push!(summary["FP"], mean(fp_vals))
        push!(summary["Power"], mean(power_vals))
        push!(summary["FDR"], mean(fdr_vals))

        println(
            "n = $n → ",
            "Power = $(round(mean(power_vals), digits=4)), ",
            "FDR = $(round(mean(fdr_vals), digits=4))"
        )
    end

    return summary, pvals_store
end

run_ttest_summary (generic function with 1 method)

In [3]:
function extract_is_DE_store(datasets)

    is_DE_store =
        Dict{Tuple{Int,Int}, Vector{Bool}}()

    for n in n_values
        for r in 1:R
            is_DE_store[(n,r)] =
                copy(datasets[(n,r)].is_DE)
        end
    end

    return is_DE_store
end

extract_is_DE_store (generic function with 1 method)

In [6]:
# =========================================================
# 2. Generate datasets for one epsilon
# =========================================================
n_values = [4, 5, 6, 8, 10, 20, 30, 40]

G = 1000
R = 100

p_signal = 0.1
β_signal = 4.0

s0_sq_true = 1.0
d0_true    = 4.0

tau_mix = 9.0

eps_values = [0.05, 0.10, 0.15]


# =========================================================
# 1. Draw mixture-Laplace errors
#
# baseline component:
#     Laplace(0, 1/sqrt(2))  -> variance 1
#
# contamination component:
#     Laplace(0, tau/sqrt(2)) -> variance tau^2
# =========================================================

function rand_mixture_laplace_errors(
    rng::AbstractRNG,
    n::Int;
    eps::Float64,
    tau::Float64
)

    b_main = 1 / sqrt(2)
    b_out  = tau / sqrt(2)

    z = Vector{Float64}(undef, n)

    @inbounds for j in 1:n

        if rand(rng) < eps
            z[j] =
                rand(
                    rng,
                    Laplace(0.0, b_out)
                )
        else
            z[j] =
                rand(
                    rng,
                    Laplace(0.0, b_main)
                )
        end
    end

    return z
end





function generate_laplace_mixture_datasets(
    eps_mix::Float64
)

    datasets =
        Dict{Tuple{Int,Int}, Any}()

    for n in n_values
        for r in 1:R

            # Same seed across epsilon settings
            # so DE indicators and sigma are matched
            rng =
                MersenneTwister(
                    10_000 + 100n + r
                )

            # -------------------------------------------------
            # DE indicators
            # -------------------------------------------------

            is_DE =
                rand(rng, G) .< p_signal


            # -------------------------------------------------
            # Gene-specific variances
            # -------------------------------------------------

            σ2 =
                rand(
                    rng,
                    InverseGamma(
                        d0_true / 2,
                        d0_true * s0_sq_true / 2
                    ),
                    G
                )

            σ =
                sqrt.(σ2)


            # -------------------------------------------------
            # Generate observations
            # -------------------------------------------------

            Y =
                Vector{Vector{Float64}}(
                    undef,
                    G
                )

            @inbounds for g in 1:G

                # Standardized signal fixed across n
                βg =
                    is_DE[g] ?
                    β_signal * σ[g] / sqrt(n) :
                    0.0

                z =
                    rand_mixture_laplace_errors(
                        rng,
                        n;
                        eps = eps_mix,
                        tau = tau_mix
                    )

                Y[g] =
                    βg .+ σ[g] .* z
            end


            datasets[(n,r)] = (
                Y = Y,
                is_DE = is_DE,
                σ = σ,
                σ2 = σ2,
                error_dist = "Laplace Mixture",
                eps_mix = eps_mix,
                tau_mix = tau_mix,
                β_signal = β_signal,
                signal_scaled_by_sqrt_n = true,
                s0_sq_true = s0_sq_true,
                d0_true = d0_true
            )
        end
    end

    return datasets
end


generate_laplace_mixture_datasets (generic function with 1 method)

In [7]:
using Distributions
using Random
using Statistics
using MultipleTesting
using JLD2

eps_values = [0.05, 0.10, 0.15]

for eps_mix in eps_values

    println("\n====================================")
    println("Running mixture Laplace eps = $eps_mix")
    println("====================================")

    # 1. Generate data
    datasets =
        generate_laplace_mixture_datasets(eps_mix)

    # 2. Store truth labels
    is_DE_store =
        extract_is_DE_store(datasets)

    # 3. Run ordinary t-test and save raw p-values
    ttest_summary, ttest_pvals_store =
        run_ttest_summary(datasets)

    # 4. Save simulation settings
    simulation_params = (
        n_values = n_values,
        G = G,
        R = R,
        p_signal = p_signal,
        β_signal = β_signal,
        s0_sq_true = s0_sq_true,
        d0_true = d0_true,
        error_dist = "Laplace Mixture",
        eps_mix = eps_mix,
        tau_mix = tau_mix,
        signal_scaled_by_sqrt_n = true
    )

    # 5. Make filename
    eps_label =
        replace(string(eps_mix), "." => "p")

    filename =
        "ttest_pvalues_laplace_mixture_eps$(eps_label).jld2"

    # 6. Save
    @save filename simulation_params ttest_summary ttest_pvals_store is_DE_store

end


Running mixture Laplace eps = 0.05

Running t-test: n = 4
n = 4 → Power = 0.0128, FDR = 0.019

Running t-test: n = 5
n = 5 → Power = 0.0681, FDR = 0.0097

Running t-test: n = 6
n = 6 → Power = 0.1691, FDR = 0.0105

Running t-test: n = 8
n = 8 → Power = 0.3162, FDR = 0.0146

Running t-test: n = 10
n = 10 → Power = 0.387, FDR = 0.0126

Running t-test: n = 20
n = 20 → Power = 0.4072, FDR = 0.0158

Running t-test: n = 30
n = 30 → Power = 0.3573, FDR = 0.0134

Running t-test: n = 40
n = 40 → Power = 0.3236, FDR = 0.0108

Running mixture Laplace eps = 0.1

Running t-test: n = 4
n = 4 → Power = 0.0089, FDR = 0.0108

Running t-test: n = 5
n = 5 → Power = 0.0452, FDR = 0.0084

Running t-test: n = 6
n = 6 → Power = 0.1024, FDR = 0.0104

Running t-test: n = 8
n = 8 → Power = 0.1831, FDR = 0.0071

Running t-test: n = 10
n = 10 → Power = 0.2244, FDR = 0.0081

Running t-test: n = 20
n = 20 → Power = 0.1887, FDR = 0.0071

Running t-test: n = 30
n = 30 → Power = 0.1234, FDR = 0.0067

Running t-test: 

In [8]:
using FastGaussQuadrature
using Distributions
using StatsFuns: logsumexp
using Optim
using MultipleTesting
using Statistics, Random, LinearAlgebra
using ApproxFun, SpecialFunctions

# =========================================================
# 1. Exact MAD-about-mean likelihood
# =========================================================

struct MADDistribution{T<:Real,S<:Integer} <: ContinuousUnivariateDistribution
    σ::T
    n::S
end

import Distributions: pdf, logpdf, insupport, minimum, maximum

function G_recursive(max_r::Int; a=0.0, b=100)
    d = Interval(a, b)
    G = Vector{Fun}(undef, max_r+1)
    G[1] = Fun(x -> 1.0, d)
    for r in 1:max_r
        integrand = Fun(x -> exp(-(x^2)/(2r*(r+1))), d) * G[r]
        G[r+1] = cumsum(integrand)
    end
    return G
end

const _Gcache = Dict{Int,Vector{Fun}}()
_getG(n) = get!(_Gcache, n) do
    G_recursive(n)
end

function _pdf_sigma1(n::Int, m::Real)
    m < 0 && return zero(float(m))
    z   = n * m / 2
    cst = n^(3/2) / (2^((n+1)/2) * π^((n-1)/2))
    G   = _getG(n)
    s   = 0.0
    @inbounds for k in 1:(n-1)
        s += binomial(n, k) * exp(-(m^2 * n^3) / (8k*(n-k))) * G[k](z) * G[n-k](z)
    end
    return cst * s
end

insupport(::MADDistribution, x::Real) = x ≥ 0
minimum(::MADDistribution) = 0.0
maximum(::MADDistribution) = Inf

pdf(d::MADDistribution, m::Real) =
    m < 0 ? 0.0 : (1 / d.σ) * _pdf_sigma1(d.n, m / d.σ)

function logpdf(d::MADDistribution, m::Real)
    m < 0 && return -Inf
    σ, n = d.σ, d.n
    z    = n * m / (2σ)
    logC = (3/2)*log(n) - ((n+1)/2)*log(2) - ((n-1)/2)*log(π)
    G    = _getG(n)

    logs = Float64[]
    @inbounds for k in 1:(n-1)
        val1 = G[k](z)
        val2 = G[n-k](z)
        if val1 > 0 && val2 > 0
            push!(logs,
                log(binomial(n, k)) -
                (m^2 * n^3) / (8σ^2 * k*(n-k)) +
                log(val1) + log(val2)
            )
        end
    end

    return -log(σ) + logC + (isempty(logs) ? -Inf : logsumexp(logs))
end


logpdf (generic function with 83 methods)

In [9]:
using FastGaussQuadrature
using Distributions
using Statistics
using MultipleTesting

# =========================================================
# Precompute quadrature for the exact MAD-normalized null
#
# Integral:
#
# p(t) = ∫ 2 Φbar(|t| c_n w) f_D(w | sigma=1,n) dw
#
# Transform:
#     w = u / (1-u),   u ∈ (0,1)
# =========================================================

function build_mad_null_quadrature(
    n::Int;
    K::Int = 400
)

    # Gauss-Legendre on [-1,1]
    x, qweights = gausslegendre(K)

    # Map to (0,1)
    u = (x .+ 1) ./ 2
    qweights ./= 2

    # Transform (0,1) -> (0,Inf)
    w_nodes = u ./ (1 .- u)

    jac = 1 ./ (1 .- u).^2

    # Exact MAD density evaluated ONLY ONCE
    mad_density = [
        pdf(
            MADDistribution(1.0, n),
            w
        )
        for w in w_nodes
    ]

    # Complete integration weights
    weights =
        qweights .* jac .* mad_density

    # Optional normalization to remove tiny numerical integration error
    weights ./= sum(weights)

    c_n =
        sqrt(pi / 2) *
        sqrt(n / (n - 1))

    return (
        w = w_nodes,
        weights = weights,
        c_n = c_n
    )
end

function mad_normalized_pvalue_fast(
    t_obs::Real,
    quad
)

    x = abs(t_obs)

    tails =
        2 .* ccdf.(
            Normal(),
            x .* quad.c_n .* quad.w
        )

    p =
        dot(
            quad.weights,
            tails
        )

    return clamp(p, 0.0, 1.0)
end

mad_quad_store =
    Dict{Int, Any}()

for n in n_values

    println(
        "Precomputing MAD null quadrature for n = $n"
    )

    @time mad_quad_store[n] =
        build_mad_null_quadrature(
            n;
            K = 400
        )
end

Precomputing MAD null quadrature for n = 4
  3.171266 seconds (30.50 M allocations: 1.548 GiB, 9.15% gc time, 98.09% compilation time)
Precomputing MAD null quadrature for n = 5
  0.028336 seconds (13.42 k allocations: 901.188 KiB)
Precomputing MAD null quadrature for n = 6
  0.010002 seconds (16.33 k allocations: 990.672 KiB)
Precomputing MAD null quadrature for n = 8
  0.094665 seconds (22.11 k allocations: 1.271 MiB)
Precomputing MAD null quadrature for n = 10
  0.064576 seconds (27.86 k allocations: 1.615 MiB)
Precomputing MAD null quadrature for n = 20
  0.219268 seconds (62.64 k allocations: 3.528 MiB)
Precomputing MAD null quadrature for n = 30
  0.326015 seconds (95.34 k allocations: 5.621 MiB)
Precomputing MAD null quadrature for n = 40
  0.620753 seconds (129.46 k allocations: 7.708 MiB, 2.68% gc time, 1.39% compilation time)


In [10]:
function run_mad_test_summary(
    datasets
)

    summary = Dict(
        "n" => Int[],
        "TP" => Float64[],
        "FP" => Float64[],
        "Power" => Float64[],
        "FDR" => Float64[]
    )

    # NEW: store raw MAD-test p-values for ranking metrics
    pvals_store =
        Dict{Tuple{Int,Int}, Vector{Float64}}()

    for n in n_values

        println(
            "\nRunning MAD test: n = $n"
        )

        quad =
            mad_quad_store[n]

        power_vals = Float64[]
        fdr_vals   = Float64[]
        tp_vals    = Float64[]
        fp_vals    = Float64[]

        for r in 1:R

            Y =
                datasets[(n,r)].Y

            is_DE =
                datasets[(n,r)].is_DE

            # -------------------------------------------------
            # Sample means
            # -------------------------------------------------

            β_hat = [
                mean(Y[g])
                for g in 1:G
            ]

            # -------------------------------------------------
            # MAD about sample mean
            # -------------------------------------------------

            mad_obs = [
                mean(
                    abs.(
                        Y[g] .- β_hat[g]
                    )
                )
                for g in 1:G
            ]

            # -------------------------------------------------
            # Normal-theory corrected MAD estimate of sigma
            # -------------------------------------------------

            sigma_hat_mad =
                quad.c_n .* mad_obs

            # -------------------------------------------------
            # MAD-normalized statistic
            # -------------------------------------------------

            t_mad =
                sqrt(n) .* β_hat ./
                sigma_hat_mad

            # -------------------------------------------------
            # Gaussian-null calibrated RAW p-values
            # -------------------------------------------------

            pvals = [
                mad_normalized_pvalue_fast(
                    t,
                    quad
                )
                for t in t_mad
            ]

            # -------------------------------------------------
            # NEW: save raw p-values
            # -------------------------------------------------

            pvals_store[(n,r)] =
                copy(pvals)

            # -------------------------------------------------
            # BH adjustment
            # -------------------------------------------------

            adj_p =
                adjust(
                    pvals,
                    BenjaminiHochberg()
                )

            reject =
                adj_p .<= 0.05

            # -------------------------------------------------
            # Metrics
            # -------------------------------------------------

            TP =
                sum(reject .& is_DE)

            FP =
                sum(reject .& .!is_DE)

            power =
                TP / max(sum(is_DE), 1)

            fdp =
                FP / max(sum(reject), 1)

            push!(tp_vals, TP)
            push!(fp_vals, FP)
            push!(power_vals, power)
            push!(fdr_vals, fdp)
        end

        push!(summary["n"], n)
        push!(summary["TP"], mean(tp_vals))
        push!(summary["FP"], mean(fp_vals))
        push!(summary["Power"], mean(power_vals))
        push!(summary["FDR"], mean(fdr_vals))

        println(
            "n = $n → ",
            "Power = $(round(mean(power_vals),digits=4)), ",
            "FDR = $(round(mean(fdr_vals),digits=4))"
        )
    end

    return summary, pvals_store
end

run_mad_test_summary (generic function with 1 method)

In [11]:
nonEB_laplace_results =
    Dict{Float64, Any}()

for eps in eps_values

    println(
        "\n\n",
        "==============================================\n",
        "       Laplace mixture: epsilon = $eps\n",
        "=============================================="
    )

    # -----------------------------------------------------
    # Generate datasets
    # -----------------------------------------------------

    datasets_eps =
        generate_laplace_mixture_datasets(
            eps
        )

    # -----------------------------------------------------
    # Store truth labels for ranking
    # -----------------------------------------------------

    is_DE_store_eps =
        Dict{Tuple{Int,Int}, Vector{Bool}}()

    for n in n_values
        for r in 1:R
            is_DE_store_eps[(n,r)] =
                copy(datasets_eps[(n,r)].is_DE)
        end
    end

    # -----------------------------------------------------
    # Ordinary t-test
    # -----------------------------------------------------

    ttest_summary_eps, ttest_pvals_store_eps =
        run_ttest_summary(
            datasets_eps
        )

    # -----------------------------------------------------
    # MAD-normalized test
    # -----------------------------------------------------

    mad_test_summary_eps, mad_test_pvals_store_eps =
        run_mad_test_summary(
            datasets_eps
        )

    # -----------------------------------------------------
    # Store in memory
    # -----------------------------------------------------

    nonEB_laplace_results[eps] = (
        ttest = ttest_summary_eps,
        mad_test = mad_test_summary_eps,
        ttest_pvals_store = ttest_pvals_store_eps,
        mad_test_pvals_store = mad_test_pvals_store_eps,
        is_DE_store = is_DE_store_eps
    )
end



       Laplace mixture: epsilon = 0.05

Running t-test: n = 4
n = 4 → Power = 0.0128, FDR = 0.019

Running t-test: n = 5
n = 5 → Power = 0.0681, FDR = 0.0097

Running t-test: n = 6
n = 6 → Power = 0.1691, FDR = 0.0105

Running t-test: n = 8
n = 8 → Power = 0.3162, FDR = 0.0146

Running t-test: n = 10
n = 10 → Power = 0.387, FDR = 0.0126

Running t-test: n = 20
n = 20 → Power = 0.4072, FDR = 0.0158

Running t-test: n = 30
n = 30 → Power = 0.3573, FDR = 0.0134

Running t-test: n = 40
n = 40 → Power = 0.3236, FDR = 0.0108

Running MAD test: n = 4
n = 4 → Power = 0.0126, FDR = 0.0204

Running MAD test: n = 5
n = 5 → Power = 0.0683, FDR = 0.0181

Running MAD test: n = 6
n = 6 → Power = 0.1758, FDR = 0.0112

Running MAD test: n = 8
n = 8 → Power = 0.3483, FDR = 0.0142

Running MAD test: n = 10
n = 10 → Power = 0.4377, FDR = 0.0133

Running MAD test: n = 20
n = 20 → Power = 0.5715, FDR = 0.0209

Running MAD test: n = 30
n = 30 → Power = 0.5934, FDR = 0.0465

Running MAD test: n = 40
n = 40 

In [13]:
for eps in eps_values

    println(
        "\n\n",
        "==============================================\n",
        "       Laplace mixture: epsilon = $eps\n",
        "=============================================="
    )

    datasets_eps =
        generate_laplace_mixture_datasets(eps)

    is_DE_store =
        Dict{Tuple{Int,Int}, Vector{Bool}}()

    for n in n_values
        for r in 1:R
            is_DE_store[(n,r)] =
                copy(datasets_eps[(n,r)].is_DE)
        end
    end

    ttest_summary, ttest_pvals_store =
        run_ttest_summary(datasets_eps)

    mad_test_summary, mad_test_pvals_store =
        run_mad_test_summary(datasets_eps)

    simulation_params = (
        n_values = n_values,
        G = G,
        R = R,
        p_signal = p_signal,
        β_signal = β_signal,
        s0_sq_true = s0_sq_true,
        d0_true = d0_true,
        error_dist = "Laplace Mixture",
        eps_mix = eps,
        tau_mix = tau_mix,
        signal_scaled_by_sqrt_n = true
    )

    eps_label =
        replace(string(eps), "." => "p")

    filename =
        "nonEB_laplace_mixture_eps$(eps_label).jld2"

    @save filename simulation_params ttest_summary mad_test_summary ttest_pvals_store mad_test_pvals_store is_DE_store
end



       Laplace mixture: epsilon = 0.05

Running t-test: n = 4
n = 4 → Power = 0.0128, FDR = 0.019

Running t-test: n = 5
n = 5 → Power = 0.0681, FDR = 0.0097

Running t-test: n = 6
n = 6 → Power = 0.1691, FDR = 0.0105

Running t-test: n = 8
n = 8 → Power = 0.3162, FDR = 0.0146

Running t-test: n = 10
n = 10 → Power = 0.387, FDR = 0.0126

Running t-test: n = 20
n = 20 → Power = 0.4072, FDR = 0.0158

Running t-test: n = 30
n = 30 → Power = 0.3573, FDR = 0.0134

Running t-test: n = 40
n = 40 → Power = 0.3236, FDR = 0.0108

Running MAD test: n = 4
n = 4 → Power = 0.0126, FDR = 0.0204

Running MAD test: n = 5
n = 5 → Power = 0.0683, FDR = 0.0181

Running MAD test: n = 6
n = 6 → Power = 0.1758, FDR = 0.0112

Running MAD test: n = 8
n = 8 → Power = 0.3483, FDR = 0.0142

Running MAD test: n = 10
n = 10 → Power = 0.4377, FDR = 0.0133

Running MAD test: n = 20
n = 20 → Power = 0.5715, FDR = 0.0209

Running MAD test: n = 30
n = 30 → Power = 0.5934, FDR = 0.0465

Running MAD test: n = 40
n = 40 